# Nightly Report: IQ, AOS, Thermal, Aberrations and Degrees of Freedom

Owner: **Guillem Megias** <br>
Last Verified to Run: **2025-07-07** <br>

In [ ]:
# Times Square Parameters
day_obs = 20251229
seq_min = 0
seq_max = 1500

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.dates import DateFormatter
from matplotlib.patches import Rectangle, Patch
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
import galsim
import numpy as np
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

def getPsfGradPerZernike(
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
    jmax: int = 22,
) -> np.ndarray:
    """Get the gradient of the PSF FWHM with respect to each Zernike.

    This function takes no positional arguments. All parameters must be passed
    by name (see the list of parameters below).

    Parameters
    ----------
    diameter : float, optional
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float, optional
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int, optional
        The minimum Noll index, inclusive. Must be >= 0. (the default is 4)
    jmax : int, optional
        The max Zernike Noll index, inclusive. Must be >= jmin.
        (the default is 22.)

    Returns
    -------
    np.ndarray
        Gradient of the PSF FWHM with respect to the corresponding Zernike.
        Units are arcsec / micron.

    Raises
    ------
    ValueError
        If jmin is negative or jmax is less than jmin
    """
    # Check jmin and jmax
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")
    if jmax < jmin:
        raise ValueError("jmax must be greater than jmin.")

    # Calculate the conversion factors
    conversion_factors = np.zeros(jmax + 1)
    for i in range(jmin, jmax + 1):
        # Set coefficients for this Noll index: coefs = [0, 0, ..., 1]
        # Note the first coefficient is Noll index 0, which does not exist and
        # is therefore always ignored by galsim
        coefs = [0] * i + [1]

        # Create the Zernike polynomial with these coefficients
        R_outer = diameter / 2
        R_inner = R_outer * obscuration
        Z = galsim.zernike.Zernike(coefs, R_outer=R_outer, R_inner=R_inner)

        # We can calculate the size of the PSF from the RMS of the gradient of
        # the wavefront. The gradient of the wavefront perturbs photon paths.
        # The RMS quantifies the size of the collective perturbation.
        # If we expand the wavefront gradient in another series of Zernike
        # polynomials, we can exploit the orthonormality of the Zernikes to
        # calculate the RMS from the Zernike coefficients.
        rms_tilt = np.sqrt(np.sum(Z.gradX.coef**2 + Z.gradY.coef**2) / 2)

        # Convert to arcsec per micron
        rms_tilt = np.rad2deg(rms_tilt * 1e-6) * 3600

        # Convert rms -> fwhm
        fwhm_tilt = 2 * np.sqrt(2 * np.log(2)) * rms_tilt

        # Save this conversion factor
        conversion_factors[i] = fwhm_tilt

    return conversion_factors[jmin:]


def convertZernikesToPsfWidth(
    zernikes: np.ndarray,
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
) -> np.ndarray:
    """Convert Zernike amplitudes to quadrature contribution to the PSF FWHM.

    Parameters
    ----------
    zernikes : np.ndarray
        Zernike amplitudes (in microns), starting with Noll index `jmin`.
        Either a 1D array of zernike amplitudes, or a 2D array, where each row
        corresponds to a different set of amplitudes.
    diameter : float
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int
        The minimum Zernike Noll index, inclusive. Must be >= 0. The
        max Noll index is inferred from `jmin` and the length of `zernikes`.
        (the default is 4, which ignores piston, x & y offsets, and tilt.)

    Returns
    -------
    dFWHM: np.ndarray
        Quadrature contribution of each Zernike vector to the PSF FWHM
        (in arcseconds).

    Notes
    -----
    Converting Zernike amplitudes to their quadrature contributions to the PSF
    FWHM allows for easier physical interpretation of Zernike amplitudes and
    the performance of the AOS system.

    For example, image we have a true set of zernikes, [Z4, Z5, Z6], such that
    ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.2, 0.3] arcsecs.
    These Zernike perturbations increase the PSF FWHM by
    sqrt[(0.1)^2 + (-0.2)^2 + (0.3)^2] ~ 0.37 arcsecs.

    If the AOS perfectly corrects for these perturbations, the PSF FWHM will
    not increase in size. However, imagine the AOS estimates zernikes, such
    that ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.3, 0.4] arcsecs.
    These estimated Zernikes, do not exactly match the true Zernikes above.
    Therefore, the post-correction PSF will still be degraded with respect to
    the optimal PSF. In particular, the PSF FWHM will be increased by
    sqrt[(0.1 - 0.1)^2 + (-0.2 - (-0.3))^2 + (0.3 - 0.4)^2] ~ 0.14 arcsecs.

    This conversion depends on a linear approximation that begins to break down
    for RSS(dFWHM) > 0.20 arcsecs. Beyond this point, the approximation tends
    to overestimate the PSF degradation. In other words, if
    sqrt(sum( dFWHM^2 )) > 0.20 arcsec, it is likely that dFWHM is
    over-estimated. However, the point beyond which this breakdown begins
    (and whether the approximation over- or under-estimates dFWHM) can change,
    depending on which Zernikes have large amplitudes. In general, if you have
    large Zernike amplitudes, proceed with caution!
    Note that if the amplitudes Z_est and Z_true are large, this is okay, as
    long as |Z_est - Z_true| is small.

    For a notebook demonstrating where the approximation breaks down:
    https://gist.github.com/jfcrenshaw/24056516cfa3ce0237e39507674a43e1

    Raises
    ------
    ValueError
        If jmin is negative
    """
    # Check jmin
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")

    # Calculate jmax from jmin and the length of the zernike array
    jmax = jmin + np.array(zernikes).shape[-1] - 1

    # Calculate the conversion factors for each zernike
    conversion_factors = getPsfGradPerZernike(
        jmin=jmin,
        jmax=jmax,
        diameter=diameter,
        obscuration=obscuration,
    )

    # Convert the Zernike amplitudes from microns to their quadrature
    # contribution to the PSF FWHM
    dFWHM = conversion_factors * zernikes

    return dFWHM

band_colors = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}
    
def annotate_bands(data: pd.DataFrame, ax: plt.Axes) -> None:
    """Anotate bottom of plot with band colors

    Parameters
    ----------
    data : pd.DataFrame
        Table of data that is being plotted
    ax : plt.Axes
        Axis on which to add annotations
    """
    # Get the bands
    bands = data['band']

    # Get the axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # Plot band bars at bottom of plot
    max_seq = data.index[-1]
    for i, band in enumerate(bands[:-1]):
        ax.plot(
            data.index[i : i + 2],
            2 * [ylim[0]],
            c=band_colors[band[0]],
            lw=7,
        )
    # Do something special for last one
    ax.plot(
        [data.index[-1], data.index[-1]+1],
        2 * [ylim[0]],
        c=band_colors[bands.iloc[-1][0]],
        lw=7,
    )

    # Restore the axis limits
    ax.set_ylim(ylim)
    ax.set_xlim(xlim)

    # Add unique bands to legend
    unique_bands = data['band'].str[0].unique()  # Get first character of each band
    
    for band_char in sorted(unique_bands):
        ax.scatter([], [], 
                  c=band_colors[band_char], 
                  s=100, 
                  marker='s', 
                  label=f'{band_char}')

def find_blocks(data: pd.DataFrame) -> pd.DataFrame:
    """Find blocks active during the night

    Parameters
    ----------
    data : pd.DataFrame
        Table of data that is being plotted
    Returns
    -------
    changes: pd.DataFrame
        dataframe containing the sequence numbers where the block changed
        and the name of the new block
    """
    block_data = data[['seq', 'block']].sort_values(by='seq')
    change_mask = block_data['block'] != block_data['block'].shift()
    block_data['block'] = block_data['block'].str.replace('BLOCK-', '', regex=False)
    blocks = pd.DataFrame({
        'seq': block_data.loc[change_mask, 'seq'],
        'block_after': block_data['block'].loc[change_mask]
    })
    return blocks

async def find_faults(client, table):
    """Find faults during the night

    Parameters
    ----------
    client: EFD client
    table : pd.DataFrame
        Table of data that is being plotted
    Returns
    -------
    table: pd.DataFrame
        incoming dataframe with the fault events added
    """
    table_time = pd.to_datetime(table['time'], format='ISO8601', utc=True)
    topics = ['MTMount', 'MTAOS', 'MTHexapod', 'MTCamera']
    table['faults'] = [None for i in range(len(table))]
    start = Time(table['obs_start'].iloc[0])
    end = Time(table['obs_end'].iloc[-1])
    for topic in topics:
        efd_data = await client.select_time_series(f"lsst.sal.{topic}.logevent_summaryState", \
                                                ['summaryState'], \
                                                 start, end)
        if len(efd_data) == 0:
            continue
        faults = efd_data[efd_data['summaryState'] == 3]
        for i in range(len(faults)):
            fault_time = pd.to_datetime(faults.index[i], format='utc')
            closest_row = table.iloc[(table_time - fault_time).abs().argmin()]
            table.loc[table['seq'] == closest_row['seq'], 'faults'] = topic
    return table


In [ ]:
from lsst.ts.xml.tables.m1m3 import *
from lsst.ts.m1m3.utils import *


async def get_m1m3_gradients(client, data):
    """Get the M1M3 thermal gradients

    Parameters
    ----------
    client: EFD client
    data : pd.DataFrame
        Table of minimal data
    Returns
    -------
    data: pd.DataFrame
        incoming dataframe with the m1m3 temperature gradients added
    """
    data_times = pd.to_datetime(data['obs_start'], format='ISO8601', utc=True)
    sorted_data_times = data_times.sort_values()
    start = Time(sorted_data_times.iloc[0])
    end = Time(sorted_data_times.iloc[-1])
    data_times = data_times.astype('int64')
    thermocouples = ThermocoupleAnalysis(client)
    await thermocouples.load(start, end, time_bin=30)
    gradients = thermocouples.xyz_r_gradients
    grad_times = pd.to_datetime(gradients.index, format='ISO8601', utc=True).astype('int64')
    t0 = grad_times[0]
    grad_times -= t0
    grad_times /=1E9
    data_times -= t0
    data_times /= 1E9
    names = ['x_gradient', 'y_gradient', 'z_gradient', 'radial_gradient']
    for name in names:
        values = gradients[name].values
        val_series = pd.Series(values)
        val_interpolated = val_series.interpolate()
        data[name] = np.interp(data_times, grad_times, val_interpolated)
    return data


In [ ]:
from matplotlib.patches import Patch
from matplotlib import cm

def plot_m1m3_gradients(data, ax):
    """Plot the M1M3 thermal gradients

    Parameters
    ----------
    data : pd.DataFrame
        Table of data to plot
    ax : matplotlib.axes._axes.Axes
        axes object on which to plot the data
    Returns
    -------
    """

    # --- Palettes: same hue family within groups, distinct shades per line ---
    blues   = plt.get_cmap('Blues')
    oranges = plt.get_cmap('Oranges')
    
    # Line colors (distinct, but clearly grouped)
    c_x = blues(0.75)     # darker blue
    c_y = blues(0.55)     # lighter blue
    c_r = oranges(0.75)   # darker orange
    c_z = oranges(0.55)   # lighter orange
    
    # Band colors (very light tint of the group hue)
    band_xy = blues(0.25)
    band_rz = oranges(0.25)
    
    # --- Shaded operational bands (put behind data) ---
    ax.axhspan(-0.4, 0.4, facecolor=band_xy, alpha=0.3, zorder=0)
    ax.axhspan(-0.1, 0.1, facecolor=band_rz, alpha=0.3, zorder=0)
    ax.axhline(0.4,  color=blues(0.65), linestyle="-", linewidth=1, alpha=0.5)
    ax.axhline(-0.4, color=blues(0.65), linestyle="-", linewidth=1, alpha=0.5)
    ax.axhline(0.1,  color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.5)
    ax.axhline(-0.1, color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.5)
    
    # --- Data lines: distinct per series with styles/markers ---
    ax.scatter(data['seq'],
        data['x_gradient'] * 8.4,
        label="X (×8.4)", color=c_x, linewidth=2.0, linestyle="-",
        s=0.5, zorder=3
    )
    ax.scatter(data['seq'],
        data['y_gradient'] * 8.4,
        label="Y (×8.4)", color=c_y, linewidth=2.0, linestyle="--",
        s=0.5, zorder=3
    )
    ax.scatter(data['seq'],
        data['radial_gradient'] * 3.7,
        label="Radial (×4.2)", color=c_r, linewidth=2.0, linestyle="-",
        s=0.5, zorder=4
    )
    ax.scatter(data['seq'],
        data['z_gradient'],
        label="Z", color=c_z, linewidth=2.0, linestyle="--",
        s=0.5, zorder=4
    )
    
    
    # --- Legends: one for data lines, one for bands (with matching colors) ---
    data_leg = ax.legend(bbox_to_anchor=(0.0, 1.25), loc='upper left',
                frameon=True, ncol=2, markerscale=4)
    ax.add_artist(data_leg)
    
    band_handles = [
        Patch(facecolor=band_xy, alpha=0.6, label="X/Y limit band (±0.4)"),
        Patch(facecolor=band_rz, alpha=0.6, label="Radial/Z limit band (±0.1)"),
    ]
    ax.legend(handles=band_handles,
              loc="upper right", frameon=True, bbox_to_anchor=(1.0, 1.25))
    
    # --- Labels, grid, cosmetics ---
    
    ax.set_ylim(-0.6, 0.6)
    return


In [ ]:
import logging

from astropy.time import Time, TimeDelta
from lsst.obs.lsst import LsstCam
from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
from lsst.summit.utils.efdUtils import (
    getEfdData,
    getMostRecentRowWithDataBefore,
    makeEfdClient,
)

from tqdm import tqdm as tqdm

__all__ = ["AOSDatabase"]


class AOSDatabase:
    table: pd.DataFrame

    def __init__(
        self,
        day_obs: int = 20250415,
        seq_min: int = 1,
        seq_max: int = 9999,
        consdb_url: str = "http://consdb-pq.consdb:8080/consdb",
    ) -> None:
        """Create fetcher.

        Parameters
        ----------
        seq_max : int, optional
            Maximum sequence number to fetch. Default is 9999.
        consdb_url : str, optional
            URL to create ConsDB client.
            The default is "http://consdb-pq.consdb:8080/consdb".
        """
        self.log = logging.getLogger(__name__)

        self.efd_client = makeEfdClient()
        self.cdb_client = ConsDbClient(consdb_url)

        self.det_order = list([191, 195, 199, 203])
        camera = LsstCam().getCamera()
        self.detector_names = [
            camera.get(det_id).getName() for det_id in self.det_order
        ]

        self.day_obs = day_obs
        self.seq_max = seq_max
        self.seq_min = seq_min
        self.table = pd.DataFrame()

        self.time_window = TimeDelta(0.2, format="sec")
        self.temp_time_window = TimeDelta(0.2, format="sec")

    async def create(self, simplified=False):
        self.table = await self._fetch(
            self.day_obs, self.seq_min, self.seq_max, simplified
        )

    async def update(self, simplified: bool = False) -> None:
        """Update the database by grabbing more recent exposures.
        This will only grab new sequences, not re-fetch existing ones.
        """
        # First grab new sequences
        seq_min = self.table["seq"].max() + 1
        updated_table = await self._fetch(
            self.day_obs, seq_min, self.seq_max, simplified=simplified
        )
        self.table = pd.concat(
            [self.table, updated_table],
            ignore_index=True,
        )
    
    async def _fetch(
        self, day_obs: int, seq_min: int, seq_max: int, simplified: bool = False
    ) -> pd.DataFrame:
        query = f"""
            SELECT
            e.air_temp AS air_temp,
            e.airmass AS airmass,
            e.dimm_seeing AS dimm,
            e.altitude AS elevation,
            e.azimuth AS azimuth,
            e.exposure_id AS visit_id,
            e.physical_filter as band,
            e.day_obs AS day_obs,
            e.exp_midpt AS time,
            e.dimm_seeing AS seeing,
            e.seq_num AS seq,
            e.science_program AS block,
            ccdvisit1_quicklook.psf_sigma,
            ccdvisit1_quicklook.z4,
            ccdvisit1_quicklook.z5,
            ccdvisit1_quicklook.z6,
            ccdvisit1_quicklook.z7,
            ccdvisit1_quicklook.z8,
            ccdvisit1_quicklook.z9,
            ccdvisit1_quicklook.z10,
            ccdvisit1_quicklook.z11,
            ccdvisit1_quicklook.z12,
            ccdvisit1_quicklook.z13,
            ccdvisit1_quicklook.z14,
            ccdvisit1_quicklook.z15,
            ccdvisit1_quicklook.z16,
            ccdvisit1_quicklook.z17,
            ccdvisit1_quicklook.z18,
            ccdvisit1_quicklook.z19,
            ccdvisit1_quicklook.z20,
            ccdvisit1_quicklook.z21,
            ccdvisit1_quicklook.z22,
            ccdvisit1_quicklook.z23,
            ccdvisit1_quicklook.z24,
            ccdvisit1_quicklook.z25,
            ccdvisit1_quicklook.z26,
            ccdvisit1_quicklook.z27,
            ccdvisit1_quicklook.z28,
            ccdvisit1.detector as detector,
            q.psf_sigma_median AS psf_fwhm,
            q.psf_sigma_min AS psf_fwhm_min,
            q.psf_sigma_max AS psf_fwhm_max,
            q.psf_area_median AS psf_area,
            q.psf_area_min AS psf_area_min,
            q.psf_area_max AS psf_area_max,
            q.aos_fwhm AS aos_fwhm,
            q.donut_blur_fwhm AS donut_blur_fwhm,
            q.physical_rotator_angle AS rotation_angle,
            e.obs_end,
            e.obs_start
            FROM
            cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
            cdb_lsstcam.ccdvisit1 AS ccdvisit1,
            cdb_lsstcam.visit1 AS visit1,
            cdb_lsstcam.visit1_quicklook AS q,
            cdb_lsstcam.exposure AS e
            WHERE
            ccdvisit1.detector IN (191, 192, 195, 196, 199, 200, 203, 204)
            AND ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
            AND ccdvisit1.visit_id = visit1.visit_id
            AND ccdvisit1.visit_id = q.visit_id
            AND ccdvisit1.visit_id = e.exposure_id
            AND (e.img_type = 'science' or e.img_type = 'acq' or e.img_type = 'engtest')
            AND e.day_obs = {day_obs}
            AND (e.seq_num BETWEEN {seq_min} AND {seq_max})
        """
        self.table = self.cdb_client.query(query).to_pandas()
        if len(self.table) == 0:
            #raise RuntimeError("ConsDB query is empty")
            return self.table

        # Correctly declare aos_fwhm and donut_blur_fwhm as float
        self.table['psf_fwhm'] = pd.to_numeric(self.table['psf_fwhm'])
        self.table['aos_fwhm'] = pd.to_numeric(self.table['aos_fwhm'])
        self.table['donut_blur_fwhm'] = pd.to_numeric(self.table['donut_blur_fwhm'])

        # Convert PSF sigma to FWHM
        sig2fwhm = 2 * np.sqrt(2 * np.log(2))
        pixel_scale = 0.2  # arcsec / pixel
        self.table["psf_fwhm"] = self.table["psf_fwhm"] * sig2fwhm * pixel_scale
        self.table["psf_fwhm_min"] = self.table["psf_fwhm_min"] * sig2fwhm * pixel_scale
        self.table["psf_fwhm_max"] = self.table["psf_fwhm_max"] * sig2fwhm * pixel_scale

        self.table["fwhm_zenith_500nm"] = [
            fwhm
            * getAirmassSeeingCorrection(airmass)
            * getBandpassSeeingCorrection(band)
            for fwhm, band, airmass in zip(
                self.table["psf_fwhm"], self.table["band"], self.table["airmass"]
            )
        ]

        zernike_columns = [f"z{i}" for i in range(4, 29)]
        self.table["zernikes"] = self.table[zernike_columns].apply(
            lambda row: np.array(row.fillna(0.0).values, dtype=float), axis=1
        )
        self.table["zernikes_fwhm"] = self.table["zernikes"].apply(
            convertZernikesToPsfWidth
        )
        
        # Get the data for the science CCDs for fwhm_05 and fwhm_95
        visits_query = f'''
        SELECT 
        ccdvisit1_quicklook.psf_sigma,
        ccdvisit1_quicklook.psf_ixx,
        ccdvisit1_quicklook.psf_iyy,
        ccdvisit1_quicklook.psf_ixy,
        ccdvisit1.detector,
        visit1.visit_id,
        visit1.seq_num AS seq,
        visit1.day_obs,
        visit1.airmass
        FROM
        cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
        cdb_lsstcam.ccdvisit1 AS ccdvisit1,
        cdb_lsstcam.visit1_quicklook AS visit1_quicklook,
        cdb_lsstcam.visit1 AS visit1 
        WHERE 
        ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
        AND ccdvisit1.visit_id = visit1.visit_id 
        AND visit1.visit_id = visit1_quicklook.visit_id
        AND ccdvisit1.detector NOT IN (168, 188, 123, 27, 0, 20, 65, 161)
        AND visit1.airmass > 0
        AND visit1.day_obs = {self.day_obs}
        AND (visit1.seq_num BETWEEN {self.seq_min} AND {self.seq_max})
        AND (visit1.img_type = 'science' or visit1.img_type = 'acq' or visit1.img_type = 'engtest')
        '''
        
        ccdvisits = self.cdb_client.query(visits_query).to_pandas()

        ccdvisits["psf_sigma"] = pd.to_numeric(ccdvisits["psf_sigma"], errors="coerce")
        ccdvisits["psf_ixx"]   = pd.to_numeric(ccdvisits["psf_ixx"], errors="coerce")
        ccdvisits["psf_iyy"]   = pd.to_numeric(ccdvisits["psf_iyy"], errors="coerce")
        ccdvisits["psf_ixy"]   = pd.to_numeric(ccdvisits["psf_ixy"], errors="coerce")

        ccdvisits["psf_fwhm"] = ccdvisits["psf_sigma"] * sig2fwhm * pixel_scale

        denom = ccdvisits["psf_ixx"] + ccdvisits["psf_iyy"]
        denom = denom.replace(0, np.nan)
        
        e1 = (ccdvisits["psf_ixx"] - ccdvisits["psf_iyy"]) / denom
        e2 = (2 * ccdvisits["psf_ixy"]) / denom
        ccdvisits['e1'] = e1
        ccdvisits['e2'] = e2
        ccdvisits["ellipticity"] = np.sqrt(e1.to_numpy()**2 + e2.to_numpy()**2)
        
        # --- Group by visit -------------------------------------------------------
        clean = ccdvisits.dropna(subset=["psf_fwhm", "ellipticity"])
        groups = clean.groupby("visit_id")        
        visits_summary = pd.DataFrame({
            'day_obs': groups['day_obs'].first(),
            'seq': groups['seq'].median(),
            'psf_fwhm_05': groups['psf_fwhm'].quantile(0.05),
            'psf_fwhm_95': groups['psf_fwhm'].quantile(0.95),
            'psf_fwhm_sigma': groups['psf_fwhm'].std(),
        
            # NEW — ellipticity min/median/max (true across CCDs)
            'psf_ellipticity_median': groups['ellipticity'].median(),
            'psf_ellipticity_sigma': groups['ellipticity'].std(),
        })
        visits_summary['psf_fwhm_95_05'] = np.sqrt(visits_summary['psf_fwhm_95']**2 - visits_summary['psf_fwhm_05']**2)
        self.table = pd.merge(
                        self.table, visits_summary, how="left", on=["seq", "day_obs"])

        self.table = await find_faults(self.efd_client, self.table)
        
        if not simplified:
            unique_day_seq = (
                self.table[["day_obs", "seq", "obs_end", "obs_start"]]
                .drop_duplicates()
                .reset_index(drop=True)
            )
            (
                days,
                seqs,
                lut,
                cam_air_temp,
                states,
                m1m3_air_temp,
                outside_temp,
                m2_air_temp,
                correction_seq,
            ) = ([] for _ in range(9))
            for idx, row in tqdm(unique_day_seq.iterrows(), total=len(unique_day_seq), disable=True):
                day_obs = int(row["day_obs"])
                seq = int(row["seq"])

                rec_end = row["obs_end"]
                rec_start = row["obs_start"]

                # ---------- Environment variables -------------
                # ----------------------------------------------
                # Get state
                # M2 and M1M3 data may be added later, but for now we 
                # just fill those DOFs with nans.
                m2_dofs_lut = np.full(20, np.nan)
                m1m3_dofs_lut = np.full(20, np.nan)
                try:
                    cam_hexapod_data = getMostRecentRowWithDataBefore(
                        self.efd_client,
                        "lsst.sal.MTHexapod.logevent_compensationOffset",
                        Time(rec_end, scale="utc"),
                        maxSearchNMinutes=10,
                        where=lambda df: df["salIndex"] == 1,
                    )

                    m2_hexapod_data = getMostRecentRowWithDataBefore(
                        self.efd_client,
                        "lsst.sal.MTHexapod.logevent_compensationOffset",
                        Time(rec_end, scale="utc"),
                        maxSearchNMinutes=10,
                        where=lambda df: df["salIndex"] == 2,
                    )

                    hexapod_val = np.array(
                        [
                            m2_hexapod_data["z"],
                            m2_hexapod_data["x"],
                            m2_hexapod_data["y"],
                            m2_hexapod_data["u"],
                            m2_hexapod_data["v"],
                            cam_hexapod_data["z"],
                            cam_hexapod_data["x"],
                            cam_hexapod_data["y"],
                            cam_hexapod_data["u"],
                            cam_hexapod_data["v"],
                        ]
                    )
                    lut_val = np.concatenate([hexapod_val, m1m3_dofs_lut, m2_dofs_lut])
                except Exception:
                    lut_val = np.full(50, np.nan)

                event = getMostRecentRowWithDataBefore(
                    self.efd_client,
                    "lsst.sal.MTAOS.logevent_degreeOfFreedom",
                    timeToLookBefore=Time(rec_start, scale="utc"),
                )
                out = np.empty(
                    50,
                )
                for i in range(50):
                    out[i] = event[f"aggregatedDoF{i}"]
                states_val = out

                seq_num_corr = event["visitId"]
                
                # Get outside temperature
                outside_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="utc"),
                    Time(rec_end, scale="utc") + self.temp_time_window,
                    index=301,
                    convert_influx_index=True
                )
                if "temperatureItem0" in outside_temp_data:
                    outside_temp_val = outside_temp_data["temperatureItem0"].mean()
                else:
                    outside_temp_val = np.nan

                # Get M2 temperature
                m2_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="utc"),
                    Time(rec_end, scale="utc") + self.temp_time_window,
                    index=112,
                    convert_influx_index=True
                )
                if "temperatureItem0" in m2_temp_data:
                    m2_air_temp_val = m2_temp_data["temperatureItem0"].mean()
                else:
                    m2_air_temp_val = np.nan

                # Get cam temperature
                cam_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="utc"),
                    Time(rec_end, scale="utc") + self.temp_time_window,
                    index=111,
                )
                if "temperatureItem0" in cam_temp_data:
                    cam_air_temp_val = cam_temp_data["temperatureItem0"].mean()
                else:
                    cam_air_temp_val = np.nan

                # Get temperature above m1m3
                m1m3_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="utc"),
                    Time(rec_end, scale="utc") + self.temp_time_window,
                    index=113,
                    convert_influx_index=True
                )
                if "temperatureItem0" in m1m3_temp_data:
                    m1m3_air_temp_val = m1m3_temp_data["temperatureItem0"].mean()
                else:
                    m1m3_air_temp_val = np.nan

                lut.append(lut_val)
                m1m3_air_temp.append(m1m3_air_temp_val)
                cam_air_temp.append(cam_air_temp_val)
                m2_air_temp.append(m2_air_temp_val)
                outside_temp.append(outside_temp_val)
                states.append(states_val)
                days.append(day_obs)
                seqs.append(seq)
                correction_seq.append(seq_num_corr)

            # Calculate FWHM Zernike contributions
            efd_table = pd.DataFrame(
                {
                    "day_obs": days,
                    "seq": seqs,
                    "cam_air_temp": cam_air_temp,
                    "m2_air_temp": m2_air_temp,
                    "m1m3_air_temp": m1m3_air_temp,
                    "outside_temp": outside_temp,
                    "lut_state": lut,
                    "dof_state": states,
                    "seq_num_corr": correction_seq,
                }
            )

            self.table = pd.merge(
                self.table, efd_table, how="left", on=["seq", "day_obs"]
            )
            m1m3_gradient_table = await get_m1m3_gradients(self.efd_client, unique_day_seq)
            self.table = pd.merge(
                self.table, m1m3_gradient_table, how="left", on=["seq", "day_obs"]
            )
            self.table["m2_delta_t"] = (
                self.table["m2_air_temp"] - self.table["m1m3_air_temp"]
            )
            self.table["dome_delta_t"] = (
                self.table["outside_temp"] - self.table["m1m3_air_temp"]
            )
            self.table["cam_m1m3_delta_t"] = (
                self.table["cam_air_temp"] - self.table["m1m3_air_temp"]
            )

        return self.table


In [ ]:

zk_groups = [[0], [11 - 4], [1, 2], [3,4], [5, 6], [22]]
zk_group_labels = ['Z4', 'Z11', 'Z5 / Z6', 'Z7 / Z8', ' Z9 / Z10', 'Z22']

groups = [[0], [5], [1, 2], [6, 7], [3, 4], [8,9]]
group_labels = [' M2 dz', 'Cam dz', 'M2 dx/dy', 'Cam dx/dy', 'M2 tilts', 'Cam tilts']
labels = ['m2 dz', 'm2 dx', 'm2 dy', 'm2 rx', 'm2 ry',
          'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry']


mirror_groups = [[10, 11, 30, 31], [12, 34], [13, 14, 32, 33], [15, 16, 35, 36], [17, 18, 37, 38]]
mirror_group_labels = ['Astig', 'Spherical', 'Trefoil', 'Coma', 'Quad']
all_labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
     'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
     '$B_{{1,1}}$', '$B_{{1,2}}$', '$B_{{1,3}}$', '$B_{{1,4}}$', '$B_{{1,5}}$',
     '$B_{{1,6}}$', '$B_{{1,7}}$', '$B_{{1,8}}$', '$B_{{1,9}}$', '$B_{{1,10}}$',
     '$B_{{1,11}}$', '$B_{{1,12}}$', '$B_{{1,13}}$', '$B_{{1,14}}$', '$B_{{1,15}}$',
     '$B_{{1,16}}$', '$B_{{1,17}}$', '$B_{{1,18}}$', '$B_{{1,19}}$', '$B_{{1,20}}$',
     '$B_{{2,1}}$', '$B_{{2,2}}$', '$B_{{2,3}}$', '$B_{{2,4}}$', '$B_{{2,5}}$',
     '$B_{{2,6}}$', '$B_{{2,7}}$', '$B_{{2,8}}$', '$B_{{2,9}}$', '$B_{{2,10}}$',
     '$B_{{2,11}}$', '$B_{{2,12}}$', '$B_{{2,13}}$', '$B_{{2,14}}$', '$B_{{2,15}}$',
     '$B_{{2,16}}$', '$B_{{2,17}}$', '$B_{{2,18}}$', '$B_{{2,19}}$', '$B_{{2,20}}$'
     ]

## Simplified Nightly Report

In [ ]:
import os, sys 
os.environ["no_proxy"] += ",.consdb"
db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
await db.create(simplified=True)
table = db.table


In [ ]:
if len(table)>0:
    filtered_table = table[table['day_obs'] == day_obs]
    raw_filtered_table = table[table['day_obs'] == day_obs]
    filtered_table = filtered_table.select_dtypes(include="number")
    filtered_table['band'] = raw_filtered_table['band']
    filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col == 'band' else 'mean' for col in filtered_table.columns})
    
    # -- AOS jumps
    aos_diff = filtered_table["aos_fwhm"].diff()
    jump_indices = filtered_table["seq"][aos_diff > 0.3].tolist()
    
    # -- States array and labels
    states_per_seq = (
        raw_filtered_table[["seq", "zernikes_fwhm"]]
        .drop_duplicates("seq")
        .dropna(subset=["zernikes_fwhm"])
        .set_index("seq")
    )
    
    
    zernikes_fwhm = np.vstack(states_per_seq["zernikes_fwhm"].values)
    seqs = states_per_seq.index.values
else:
    print(f'No  data available for {day_obs},  seq_nums {seq_min}-{seq_max}')

In [ ]:
if len(table)>0:
    fig = plt.figure(figsize=(30, 15))

    linewidth = 0.7
    # Top 5x4 GridSpec occupies the upper half
    gs_top = gridspec.GridSpec(
        nrows=5, ncols=4,
        width_ratios=[4, 2, 2, 2],
        height_ratios=[1]*5,
        hspace=0.0,
        wspace=0.25,
        top=0.97,
        bottom=0.54  # ends halfway down
    )
    
    # Bottom 5x4 GridSpec occupies the lower half
    gs_bot = gridspec.GridSpec(
        nrows=5, ncols=4,
        width_ratios=[4, 2, 2, 2],
        height_ratios=[1]*5,
        hspace=0.0,
        top=0.46,
        bottom=0.05
    )
    
    # -- Left column: Survey performance
    axes = []
    for i in range(5):
        ax = fig.add_subplot(gs_top[i, 0], sharex=axes[0] if i > 0 else None)
        axes.append(ax)
    axes[0].scatter(filtered_table['seq'], filtered_table['fwhm_zenith_500nm'], s=3, label='FWHM_Zenith_500nm')
    axes[0].scatter(filtered_table['seq'], filtered_table['donut_blur_fwhm'], s=3, label='Donut Blur FWHM')
    axes[0].scatter(filtered_table['seq'], filtered_table["psf_fwhm_95_05"], s=3, label="FWHM:sqrt(95^2-5^2)")
    #axes[0].scatter(filtered_table['seq'], filtered_table["aos_fwhm"], s=3, label="AOS FWHM")
    try:
        axes[0].scatter(filtered_table['seq'], filtered_table['dimm'], s=3, label='DIMM')
    except KeyError:
        pass
    axes[0].legend(fontsize=8, bbox_to_anchor=(1.0, 1.5), loc='upper right', markerscale=2)
    ymin0 = 0; ymax0 = 1.5
    axes[0].set_ylim(ymin0, ymax0)
    (xmin, xmax) = axes[0].get_xlim()
    axes[0].text(xmin, ymax0 * 1.1, "... Faults", fontsize=14, color='magenta')
    axes[0].set_ylabel('FWHM [arcsec]')
    axes[0].set_title(f'Delivered Seeing and System Variables')
    axes[1].scatter(filtered_table['seq'], filtered_table["aos_fwhm"], s=3, label="AOS FWHM")
    axes[1].scatter(filtered_table['seq'], filtered_table["psf_fwhm_95_05"], s=3, label="FWHM:sqrt(95^2-5^2)")
    axes[1].set_ylim(ymin0, ymax0)
    axes[2].scatter(filtered_table['seq'], filtered_table['elevation'], color='k', s=3)
    axes[3].scatter(filtered_table['seq'], filtered_table['azimuth'], color='k', s=3)
    axes[4].scatter(filtered_table['seq'], filtered_table['rotation_angle'], color='k', s=3)
    
    axes[0].set_ylabel('FWHM\n[arcsec]')
    axes[1].set_ylabel('AOS FWHM\n[arcsec]')
    axes[2].set_ylabel('Elevation\n[deg]')
    axes[3].set_ylabel('Azimuth\n[deg]')
    axes[4].set_ylabel('Rotation\n[deg]')
    axes[4].set_xlabel('Sequence Number')

    leg1 = axes[1].legend(loc='upper center', ncols=2, markerscale=2, framealpha=0.5)
    leg1.get_frame().set_linewidth(0)
    leg1.get_frame().set_edgecolor("none")
    leg1.get_frame().set_facecolor("none")

    annotate_bands(filtered_table, axes[4])
    axes[4].legend(ncols=3)
    
    
    for ax in axes[1::]:
        for x in jump_indices:
            ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
    # Add block boundaries and names
    blocks = find_blocks(table) # Find active blocks
    maxSeq = max(table['seq'].values)
    for i, changeSeq in enumerate(blocks['seq']):
        if i < len(blocks) - 1:
            changeWidth = blocks['seq'].iloc[i+1] - changeSeq
        else:
            changeWidth = maxSeq - changeSeq
        for ax in axes[0::]:
            ax.axvline(changeSeq, ls = '--', color='k', linewidth=linewidth, alpha=0.5)
        if changeWidth > 5:
            (ymin, ymax) = axes[1].get_ylim()
            ytext = ymin = (ymax - ymin) * 0.5
            axes[1].text(int(changeSeq + changeWidth / 2 - 2), ytext, 
                         blocks['block_after'].iloc[i], fontsize=8, rotation=90, alpha=0.6) 
    # Now plot the faults
    faults = table[table['faults'].notnull()] 
    for k in range(len(faults)):
        for ax in axes[0::]:
            ax.axvline(faults['seq'].iloc[k], color='magenta', ls=':')
        
    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
    for ax in axes:
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction='in', which='both')
    
    
    # -- Right column: DoF plots with shared x-axis (not y)
    axes = [fig.add_subplot(gs_top[i, 1]) for i in range(5)]
    for id_group, (ax, zk_group) in enumerate(zip(axes, zk_groups)):
        zk_labels = zk_group_labels[id_group]
        for zk_idx, i in enumerate(zk_group):
            if len(zk_group) == 1:
                zk_label = zk_labels
            if len(zk_group) == 2:
                zk_label = zk_labels.split('/')[zk_idx].strip()
            color = 'black' if zk_idx == 0 else 'gray' if zk_idx == 1 else None
            vals = zernikes_fwhm[:, i]
            ax.scatter(seqs, vals, s=5, color=color, label=zk_label)
        ax.set_ylabel(f"{zk_group_labels[id_group]}\n[arcsec]")
        if zk_group_labels[id_group] == 'Z4':
            # This line is showing the recently added focus offset
            ax.axhline(-0.20, ls='--', color='green')
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction="in")
        leg = ax.legend(bbox_to_anchor=(1.22, 0.5), loc='center right')
        leg.get_frame().set_linewidth(0)
        leg.get_frame().set_edgecolor("none")
    
    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
        ax.grid(True, alpha=0.5)
    for ax in axes:
        for x in jump_indices:
            ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
    axes[-1].set_xlabel("Sequence Number")
    axes[0].set_title(f'Optical Aberrations')
    
    fig.suptitle("Survey Mode Performance Simplified – Day " + str(day_obs), fontsize=18, x=0.35, y=1.03)
    
else:
    print('No data to plot')

fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/Simple_Night_Report_Alt2_{day_obs}.png", 
            bbox_inches='tight', pad_inches=1.2)    

## Full Nightly Report

In [ ]:
day_obs = 20260104
db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
await db.create(simplified=False)
table = db.table


In [ ]:
if len(table)>0:
    filtered_table = table[table['day_obs'] == day_obs]
    raw_filtered_table = table[table['day_obs'] == day_obs]
    filtered_table = filtered_table.select_dtypes(include="number")
    filtered_table['band'] = raw_filtered_table['band']
    filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col == 'band' else 'mean' for col in filtered_table.columns})
    
    # -- AOS jumps
    aos_diff = filtered_table["aos_fwhm"].diff()
    jump_indices = filtered_table["seq"][aos_diff > 0.3].tolist()
    
    # -- States array and labels
    states_per_seq = (
        raw_filtered_table[["seq", "dof_state", "zernikes_fwhm", "lut_state"]]
        .drop_duplicates("seq")
        .dropna(subset=["dof_state", "zernikes_fwhm", "lut_state"])
        .set_index("seq")
    )
    
    dof_state = np.vstack(states_per_seq["dof_state"].values)
    zernikes_fwhm = np.vstack(states_per_seq["zernikes_fwhm"].values)
    lut_state = np.vstack(states_per_seq["lut_state"].values)
    seqs = states_per_seq.index.values
else:
    print(f'No  data available for {day_obs},  seq_nums {seq_min}-{seq_max}')

filtered_table_04 = filtered_table
dof_state_04 = dof_state
zernikes_fwhm_04 = zernikes_fwhm
lut_state_04 = lut_state
seqs_04 = seqs

In [ ]:
day_obs = 20251217
db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
await db.create(simplified=False)
table = db.table


In [ ]:
if len(table)>0:
    filtered_table = table[table['day_obs'] == day_obs]
    raw_filtered_table = table[table['day_obs'] == day_obs]
    filtered_table = filtered_table.select_dtypes(include="number")
    filtered_table['band'] = raw_filtered_table['band']
    filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col == 'band' else 'mean' for col in filtered_table.columns})
    
    # -- AOS jumps
    aos_diff = filtered_table["aos_fwhm"].diff()
    jump_indices = filtered_table["seq"][aos_diff > 0.3].tolist()
    
    # -- States array and labels
    states_per_seq = (
        raw_filtered_table[["seq", "dof_state", "zernikes_fwhm", "lut_state"]]
        .drop_duplicates("seq")
        .dropna(subset=["dof_state", "zernikes_fwhm", "lut_state"])
        .set_index("seq")
    )
    
    dof_state = np.vstack(states_per_seq["dof_state"].values)
    zernikes_fwhm = np.vstack(states_per_seq["zernikes_fwhm"].values)
    lut_state = np.vstack(states_per_seq["lut_state"].values)
    seqs = states_per_seq.index.values
else:
    print(f'No  data available for {day_obs},  seq_nums {seq_min}-{seq_max}')

filtered_table_17 = filtered_table
dof_state_17 = dof_state
zernikes_fwhm_17 = zernikes_fwhm
lut_state_17 = lut_state
seqs_17 = seqs

In [ ]:
print(len(seqs_04), len(seqs_17))

In [ ]:
camera_contrib = 0.207 # From lab measurements at SLAC
for [day_obs, filtered_table] in [[20260104, filtered_table_04], [20251217, filtered_table_17]]:
    sys_2 = filtered_table["aos_fwhm"].values
    plt.hist(sys_2, bins=30, range=(0.0, 1.0), histtype='step', lw=2, label=day_obs)
    plt.xlim(0.0, 1.0)
plt.legend()
plt.title("AOS_FWHM 20260104 vs 20251217")
plt.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/Compare_1_0104_1217.png")

In [ ]:
index = 100
seq = int(seqs[index])
aos = filtered_table[filtered_table['seq'] == seq]['aos_fwhm'].values[0]
print(aos)
zer = zernikes_fwhm[index,:]
zsum = sum(val * val for val in zer)
z4 = zer[0]
#zsum -= z4 * z4
#zsum += (z4+0.2) * (z4+0.2)
qsum = np.sqrt(zsum)
#qsum = 1.06 * np.log(1.0 + qsum)
print(zer, qsum, aos, seq)

In [ ]:
fig = plt.figure(figsize=(12,12))
gs_top = gridspec.GridSpec(
    nrows=6, ncols=1,
    width_ratios=[4],
    height_ratios=[1]*6,
    hspace=0.0,
    top=0.97,
    bottom=0.03  # ends halfway down
)
axes = [fig.add_subplot(gs_top[i]) for i in range(6)]

for [day_obs, seqs, zernikes_fwhm] in [[20260104, seqs_04, zernikes_fwhm_04], [20251217, seqs_17, zernikes_fwhm_17]]:
        for id_group, (ax, zk_group) in enumerate(zip(axes, zk_groups)):
            zk_labels = zk_group_labels[id_group]
            for zk_idx, i in enumerate(zk_group):
                if len(zk_group) == 1:
                    zk_label = zk_labels
                if len(zk_group) == 2:
                    zk_label = zk_labels.split('/')[zk_idx].strip()
                #color = 'black' if zk_idx == 0 else 'gray' if zk_idx == 1 else None
                vals = zernikes_fwhm[:, i]
                ax.scatter(seqs, vals, s=5,  label=zk_label+str(day_obs)+f" med={np.nanmedian(vals):.2f}")
            ax.set_ylabel(f"{zk_group_labels[id_group]}\n[arcsec]")
            if zk_group_labels[id_group] == 'Z4':
                # This line is showing the recently added focus offset
                ax.axhline(-0.20, ls='--', color='green')
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction="in")
            ax.set_ylim(-1.0, 1.0)
            leg1 = ax.legend(bbox_to_anchor=(1.25, 0.5), loc='center right', markerscale=2)
            leg1.get_frame().set_linewidth(0)
            leg1.get_frame().set_edgecolor("none")
            leg1.get_frame().set_facecolor((1, 1, 1, 0))

fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/Compare_2_0104_1217.png", bbox_inches='tight', pad_inches=1.2)


In [ ]:
for i in range(23):
    for [day_obs, seqs, zernikes_fwhm] in [[20251216, seqs_16, zernikes_fwhm_16], [20251217, seqs_17, zernikes_fwhm_17]]:
        vals = zernikes_fwhm[:, i]
        print(i+4, day_obs, f" {np.nanmedian(vals):.2f}")
    print()



In [ ]:
states_per_seq.head(5)

In [ ]:
for [day_obs, seqs, zernikes_fwhm, filtered_table] in [[20251216, seqs_16, zernikes_fwhm_16, filtered_table_16], [20251217, seqs_17, zernikes_fwhm_17, filtered_table_17]]:
    residuals = []
    aos_fwhm = []
    for i, seq in enumerate(seqs):
        aos = filtered_table[filtered_table['seq'] == seq]['aos_fwhm'].values[0]
        zsum = sum(val * val for val in zernikes_fwhm[i, :])
        z4 = zernikes_fwhm[i, 0]
        #zsum -= z4 * z4
        #zsum += (z4+0.2) * (z4+0.2)
        qsum = np.sqrt(zsum)
        qsum = 1.06 * np.log(1.0 + qsum)
        residuals.append(qsum)
        aos_fwhm.append(aos)
    plt.hist(residuals, bins=30, range=(0.0, 1.0), histtype='step', lw=2, label=day_obs)
plt.legend()

In [ ]:
for [day_obs, seqs, zernikes_fwhm, filtered_table] in [[20251216, seqs_16, zernikes_fwhm_16, filtered_table_16], [20251217, seqs_17, zernikes_fwhm_17, filtered_table_17]]:
    residuals = []
    aos_fwhm = []
    for i, seq in enumerate(seqs):
        aos = filtered_table[filtered_table['seq'] == seq]['aos_fwhm'].values[0]
        zsum = sum(val * val for val in zernikes_fwhm[i, :])
        z4 = zernikes_fwhm[i, 0]
        #zsum -= z4 * z4
        #zsum += (z4+0.2) * (z4+0.2)
        qsum = np.sqrt(zsum)
        qsum = 1.06 * np.log(1.0 + qsum)
        residuals.append(qsum)
        aos_fwhm.append(aos)
    plt.scatter(aos_fwhm, residuals, label=day_obs)
plt.legend()
plt.ylim(0.25, 1.0)
plt.xlim(0.25, 1.0)
plt.xlabel("Tabulated AOS_FWHM")
plt.ylabel("Zernike QuadSum, with 1.06*np.log(1.0+qsum) correction")
plt.title("AOS_FWHM vs Zernike quad sum")
xs = np.linspace(0.25, 1.0, 100)
plt.plot(xs, xs, ls='--', color='red')
plt.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/AOS_FWHM_vs_Zernike_QuadSum_30Dec25.png")

In [ ]:
filtered_table = filtered_table_17
index = 3
seq = seqs[index]
aos = filtered_table[filtered_table['seq'] == seq]['aos_fwhm'].values[0]
zsum = sum(val * val for val in zernikes_fwhm[index, :])
z4 = zernikes_fwhm[index, 0]
zsum -= z4 * z4
zsum += (z4+0.2) * (z4+0.2)
qsum = np.sqrt(zsum)
qsum = 1.06 * np.log(1.0 + qsum)
print(aos, qsum, z4)

In [ ]:
0.4574153001752258 0.5043964282508856

In [ ]:
z4

In [ ]:
max_corr_lag = 10
for [day_obs, seqs, zernikes_fwhm, filtered_table] in [[20251216, seqs_16, zernikes_fwhm_16, filtered_table_16], [20251217, seqs_17, zernikes_fwhm_17, filtered_table_17]]:
    corr_lag = filtered_table['seq'] - filtered_table['seq_num_corr']
    corr_lag = np.clip(corr_lag, 0, max_corr_lag)
    print(day_obs, np.nanmean(corr_lag))

In [ ]:
import lsst.summit.utils.butlerUtils as butlerUtils
butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=False)


In [ ]:
from lsst.summit.utils.butlerUtils import getExpRecordFromDataId
expId = 2025121600116
dataId = {'exposure':expId, 'instrument':'LSSTCam'}
expRecord = getExpRecordFromDataId(butler, dataId)

In [ ]:
dataId = {'visit':expId, 'instrument':'LSSTCam'}
zer = butler.get("aggregateZernikesAvg", dataId=dataId)

In [ ]:
if len(table)>0:
    fig = plt.figure(figsize=(31, 18))
    
    linewidth = 0.7
    # Top 5x4 GridSpec occupies the upper half
    gs_top = gridspec.GridSpec(
        nrows=6, ncols=4,
        width_ratios=[4, 2, 2.15, 2],
        height_ratios=[1]*6,
        hspace=0.0,
        wspace=0.26,
        top=0.97,
        bottom=0.54  # ends halfway down
    )
    
    # Bottom 5x4 GridSpec occupies the lower half
    gs_bot = gridspec.GridSpec(
        nrows=6, ncols=4,
        width_ratios=[4, 2, 2.15, 2],
        height_ratios=[1]*6,
        hspace=0.0,
        wspace=0.26,
        top=0.46,
        bottom=0.05
    )
    
    # -- Left column: Survey performance
    axes = []
    for i in range(6):
        ax = fig.add_subplot(gs_top[i, 0], sharex=axes[0] if i > 0 else None)
        axes.append(ax)
    axes[0].scatter(filtered_table['seq'], filtered_table['fwhm_zenith_500nm'], s=3, label='FWHM_Zenith_500nm')
    axes[0].scatter(filtered_table['seq'], filtered_table['donut_blur_fwhm'], s=3, label='Donut Blur FWHM')
    axes[0].scatter(filtered_table['seq'], filtered_table["psf_fwhm_95_05"], s=3, label="FWHM:sqrt(95^2-5^2)")
    #try:
    #    axes[0].scatter(filtered_table['seq'], filtered_table['dimm'], s=3, label='DIMM')
    #except KeyError:
    #    pass
    axes[0].legend(fontsize=8, bbox_to_anchor=(1.0, 1.6), loc='upper right', markerscale=2)
    ymin0 = 0; ymax0 = 1.5
    axes[0].set_ylim(ymin0, ymax0)
    (xmin, xmax) = axes[0].get_xlim()
    axes[0].text(xmin, ymax0 * 1.1, "... Faults", fontsize=14, color='magenta')
    axes[0].set_ylabel('FWHM [arcsec]')
    axes[0].set_title(f'Delivered Seeing and System Variables')
    axes[1].scatter(filtered_table['seq'], filtered_table["aos_fwhm"], s=3, label='AOS FWHM')
    axes[2].scatter(filtered_table['seq'], filtered_table['elevation'], color='k', s=3)
    axes[3].scatter(filtered_table['seq'], filtered_table['azimuth'], color='k', s=3)
    axes[4].scatter(filtered_table['seq'], filtered_table['rotation_angle'], color='k', s=3)
    max_corr_lag = 10
    corr_lag = filtered_table['seq'] - filtered_table['seq_num_corr']
    corr_lag = np.clip(corr_lag, 0, max_corr_lag)
    axes[5].scatter(filtered_table['seq'], corr_lag, color='k', s=3)
    
    axes[0].set_ylabel('FWHM\n[arcsec]')
    axes[1].set_ylabel('AOS FWHM\n[arcsec]')
    axes[2].set_ylabel('Elevation\n[deg]')
    axes[3].set_ylabel('Azimuth\n[deg]')
    axes[4].set_ylabel('Rotator\n[deg]')
    axes[5].set_ylabel('Correction Lag\n[# of visits]')
    axes[5].set_xlabel('Sequence Number')
    
    annotate_bands(filtered_table, axes[5])
    axes[5].legend(ncols=3, fontsize=8)
    
    
    for ax in axes[1::]:
        for x in jump_indices:
            ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
    # Add block boundaries and names
    blocks = find_blocks(table) # Find active blocks
    maxSeq = max(table['seq'].values)
    for i, changeSeq in enumerate(blocks['seq']):
        if i < len(blocks) - 1:
            changeWidth = blocks['seq'].iloc[i+1] - changeSeq
        else:
            changeWidth = maxSeq - changeSeq
        for ax in axes[0::]:
            ax.axvline(changeSeq, ls = '--', color='k', linewidth=linewidth, alpha=0.5)
        if changeWidth > 5:
            (ymin1, ymax1) = axes[1].get_ylim()
            ytext = ymin1 = (ymax1 - ymin1) * 0.5
            axes[1].text(int(changeSeq + changeWidth / 2 - 2), ytext, 
                         blocks['block_after'].iloc[i], fontsize=8, rotation=90, alpha=0.6)    

    # Now plot the faults
    faults = table[table['faults'].notnull()] 
    for k in range(len(faults)):
        for ax in axes[0::]:
            ax.axvline(faults['seq'].iloc[k], color='magenta', ls=':')

    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
    for ax in axes:
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction='in', which='both')
    
    # Thermal gradients 
    axes[0] = fig.add_subplot(gs_bot[0:2, 0])
    axes[1] = fig.add_subplot(gs_bot[2:4, 0])
    axes[2] = fig.add_subplot(gs_bot[4:, 0])
    axes = [axes[0], axes[1], axes[2]]
    plot_m1m3_gradients(filtered_table, axes[0])
    min_temp = 1000.0; max_temp = -1000
    for temp in ['cam_air_temp', 'm2_air_temp', 'm1m3_air_temp', 'outside_temp']:
        this_max = np.max(filtered_table[temp])
        this_min = np.min(filtered_table[temp])
        if  this_max > max_temp:
            max_temp = this_max
        if  this_min < min_temp:
            min_temp = this_min
    axes[1].scatter(filtered_table['seq'], filtered_table['cam_air_temp'], color="C0", s=3, label="ESS:111 camera air Temperature")
    axes[1].scatter(filtered_table['seq'], filtered_table['m2_air_temp'], color="C1", s=3, label="ESS:112 M2 air Temperature")
    axes[1].scatter(filtered_table['seq'], filtered_table['m1m3_air_temp'], color="C2", s=3, label="ESS:113 M1M3 air Temperature")
    axes[1].scatter(filtered_table['seq'], filtered_table['outside_temp'], color="C6", s=3, label="ESS:301 Outside Temperature")
    leg4 = axes[1].legend(loc='upper right', ncols=2, markerscale=2)
    leg4.get_frame().set_linewidth(0)
    leg4.get_frame().set_edgecolor("none")
    leg4.get_frame().set_facecolor("none")

    axes[1].set_ylim(min_temp * 0.95, max_temp * 1.1)
    
    axes[2].scatter(filtered_table['seq'], filtered_table['m2_delta_t'], color='C0', s=3, label='M2-M1M3')
    axes[2].scatter(filtered_table['seq'], filtered_table['cam_m1m3_delta_t'], color='C1', s=3, label='Cam-M1M3')
    axes[2].scatter(filtered_table['seq'], filtered_table['dome_delta_t'], color='C2', s=3, label='Out-M1M3')
    leg3 = axes[2].legend(loc='upper right', ncols=3, markerscale=2, framealpha=0.5)
    leg3.get_frame().set_linewidth(0)
    leg3.get_frame().set_edgecolor("none")
    leg3.get_frame().set_facecolor("none")

    
    axes[1].set_ylabel('Temps\n[C]')
    axes[2].set_ylabel('Temp DIffs\n[C]')
    axes[2].set_xlabel('Sequence Number')
    
    for ax in axes:
        for x in jump_indices:
            ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
    for ax in axes:
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction='in', which='both')
    axes[0].set_title(f'M1M3 Thermal Gradients', y=1.02)
    
    # -- Right column: DoF plots with shared x-axis (not y)
    axes = [fig.add_subplot(gs_top[i, 1]) for i in range(6)]
    for id_group, (ax, zk_group) in enumerate(zip(axes, zk_groups)):
        zk_labels = zk_group_labels[id_group]
        for zk_idx, i in enumerate(zk_group):
            if len(zk_group) == 1:
                zk_label = zk_labels
            if len(zk_group) == 2:
                zk_label = zk_labels.split('/')[zk_idx].strip()
            color = 'black' if zk_idx == 0 else 'gray' if zk_idx == 1 else None
            vals = zernikes_fwhm[:, i]
            ax.scatter(seqs, vals, s=5, color=color, label=zk_label)
        ax.set_ylabel(f"{zk_group_labels[id_group]}\n[arcsec]")
        if zk_group_labels[id_group] == 'Z4':
            # This line is showing the recently added focus offset
            ax.axhline(-0.20, ls='--', color='green')
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction="in")
        ax.set_ylim(-1.0, 1.0)
        leg1 = ax.legend(bbox_to_anchor=(1.22, 0.5), loc='center right', markerscale=2)
        leg1.get_frame().set_linewidth(0)
        leg1.get_frame().set_edgecolor("none")
        leg1.get_frame().set_facecolor((1, 1, 1, 0))
    
    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
        ax.grid(True, alpha=0.5)
    for ax in axes:
        for x in jump_indices:
            ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
    axes[-1].set_xlabel("Sequence Number")
    axes[0].set_title(f'Optical Aberrations')
    
    
    # -- Center column: DoF plots with shared x-axis (not y)
    axes = [fig.add_subplot(gs_bot[i, 1]) for i in range(6)]
    for id_group, (ax, dof_group) in enumerate(zip(axes, groups)):
        for i in dof_group:
            vals = (lut_state[:, i] + dof_state[:, i])
            ax.scatter(seqs, vals, label=f"{labels[i]}", s=3)
        ax.set_ylabel(group_labels[id_group])
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction="in")
        leg2 = ax.legend(bbox_to_anchor=(1.28, 0.5), loc='center right', markerscale=3)
        leg2.get_frame().set_linewidth(0)
        leg2.get_frame().set_edgecolor("none")
        leg2.get_frame().set_facecolor("none")

    
    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
        ax.grid(True, alpha=0.5)
    for ax in axes:
        for x in jump_indices:
            ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
    axes[0].set_title(f'Hexapods State (LUT + trim)')
    axes[-1].set_xlabel("Sequence Number")


    # -- Right column: System contribution plots
    ax1 = fig.add_subplot(gs_top[3:, 2])
    ax0 = fig.add_subplot(gs_top[:3, 2], sharex=ax1)
    ax3 = fig.add_subplot(gs_bot[3:, 2])
    ax2 = fig.add_subplot(gs_bot[:3, 2], sharex=ax3)
    ax5 = fig.add_subplot(gs_bot[3:, 3])
    ax4 = fig.add_subplot(gs_bot[:3, 3], sharex=ax5)
    axes = [ax0, ax1, ax2, ax3, ax4, ax5]

    axes[0].scatter(filtered_table['psf_fwhm'], filtered_table['donut_blur_fwhm'], s=4)
    axes[0].set_xlim(0.4, 2.0)
    axes[0].set_ylim(0.4, 2.0)
    axes[0].set_ylabel("Donut blur [arcsec]")
    axes[0].set_title("Image Quality Metrics")
    axes[0].set_aspect(1.0)
    axes[0].tick_params(labelbottom=False)
    axes[0].grid(alpha=0.2)
    
    xs = np.linspace(0.4, 2.0, 100)
    axes[0].plot(xs, xs, ls='--', color='red')

    donut_blur = filtered_table['donut_blur_fwhm'].values
    ax0_histy = axes[0].inset_axes([1.0, 0, 0.5, 1], sharey=axes[0])
    ax0_histy.hist(donut_blur, bins=20, histtype='step', lw=2, orientation='horizontal')
    ax0_histy.yaxis.tick_right()
    ax0_histy.set_xticks([])
    ax0_histy.set_xticklabels([])
    ax0_histy.grid(alpha=0.2)
    ax0_histy.text(1, 1.9, f'Median = {np.nanmedian(donut_blur):.2f}"')

    x  = filtered_table["psf_fwhm"].values
    y  = filtered_table["psf_ellipticity_median"].values
    xerr = filtered_table["psf_fwhm_sigma"].values
    yerr = filtered_table["psf_ellipticity_sigma"].values
    axes[1].errorbar(
        x, y,
        xerr=xerr,
        yerr=yerr,
        fmt='o', markersize=3,
        elinewidth=0.1, capsize=1,
        color='tab:blue', alpha=0.7
    )
    axes[1].set_ylabel("ellipticity")
    axes[1].set_xlabel("FWHM [arcsec]")
    axes[1].set_xticks([0.6, 1.0, 1.4, 1.8, 2.0])
    axes[1].set_box_aspect(1.0)
    axes[1].grid(alpha=0.2)

    ax1_histy = axes[1].inset_axes([1.0, 0, 0.5, 1], sharey=axes[1])
    ax1_histy.hist(y, bins=20, orientation='horizontal', histtype='step', lw=2)
    ax1_histy.yaxis.tick_right()
    ax1_histy.grid(alpha=0.2)
    ax1_histy.text(1, 0.5, f"Median = {np.nanmedian(y):.2f}")

    camera_contrib = 0.207 # From lab measurements at SLAC
    x  = np.sqrt(filtered_table["psf_fwhm"].values**2 - (filtered_table["donut_blur_fwhm"].values**2 - camera_contrib**2))
    y  = filtered_table["psf_ellipticity_median"].values
    xerr = filtered_table["psf_fwhm_sigma"].values
    in_spec = [[a,b] for a,b in zip(x,y) if (a < 0.45) and (b < 0.1)]
    pass_2 = f"\n{(len(in_spec) / len(x) * 100.0):.1f} % in green region"
    axes[2].scatter(x, y, s=4, color='tab:orange')
    axes[2].set_ylabel("ellipticity")
    axes[2].set_box_aspect(1.0)
    axes[2].tick_params(labelbottom=False)
    axes[2].grid(alpha=0.2)
    axes[2].set_title("System Contribution Metrics")
    axes[2].set_ylim(top = 0.5)

    x  = filtered_table["psf_fwhm_95_05"]
    y  = filtered_table["psf_ellipticity_median"].values
    xerr = filtered_table["psf_fwhm_sigma"].values
    in_spec = [[a,b] for a,b in zip(x,y) if (a < 0.45) and (b < 0.1)]
    pass_3 = f"\n{(len(in_spec) / len(x) * 100.0):.1f} % in green region"
    axes[3].scatter(x, y, s=4, color='tab:red')
    axes[3].set_ylabel("ellipticity")
    axes[3].set_xlabel("[arcsec]")
    axes[3].set_box_aspect(1.0)
    axes[3].grid(alpha=0.2)
    axes[3].set_ylim(top = 0.5)

    sys_2 = np.sqrt(filtered_table["aos_fwhm"].values**2 + camera_contrib**2)
    sys_3 = np.sqrt(filtered_table["psf_fwhm"].values**2 - (filtered_table["donut_blur_fwhm"].values**2 - camera_contrib**2))
    axes[4].hist(sys_2, bins=30, range=(0.0, 1.0), histtype='step', color='tab:blue', lw=2, label='sqrt(AOS^2+Cam^2)')
    axes[4].hist(sys_3, bins=30, range=(0.0, 1.0), histtype='step', color='tab:orange', lw=2, label='sqrt(FWHM^2-(Blur^2-Cam^2))')
    axes[4].hist(filtered_table["psf_fwhm_95_05"], bins=30, range=(0.0, 1.0), histtype='step', lw=2, color='tab:red', label='FWHM:sqrt(95^2-5^2)')
    axes[4].set_xlim(0.0, 1.0)
    axes[4].set_box_aspect(1.0)
    axes[4].tick_params(labelbottom=False)
    axes[4].axvline(0.45, ls='--', color='black', alpha=0.5)
    legend_patches = [
        Patch(facecolor='tab:blue',   edgecolor='none', label='sqrt(AOS^2 + Cam^2)'),
        Patch(facecolor='tab:orange',    edgecolor='none', label='FWHM: sqrt(95^2 - 5^2)'),
        Patch(facecolor='tab:red',    edgecolor='none', label='sqrt(FWHM^2 - (Blur^2 - Cam^2))'),
    ]
    
    axes[4].legend(
        handles=legend_patches,
        loc='upper center',
        bbox_to_anchor=(0.4, 1.2),
        frameon=False,
        borderaxespad=0.,
    )
    shift = -0.035
    pos = axes[4].get_position()
    axes[4].set_position([pos.x0 + shift, pos.y0, pos.width, pos.height])
    axes[4].grid(alpha=0.2)

    x  = sys_2
    y  = filtered_table["psf_ellipticity_median"].values
    xerr = filtered_table["psf_fwhm_sigma"].values
    in_spec = [[a,b] for a,b in zip(x,y) if (a < 0.45) and (b < 0.1)]
    pass_5 = f"\n{(len(in_spec) / len(x) * 100.0):.1f} % in green region"
    axes[5].scatter(x, y, s=4, color='tab:blue')
    axes[5].set_xlabel("[arcsec]")
    axes[5].set_box_aspect(1.0)
    shift = -0.035
    pos = axes[5].get_position()
    axes[5].set_position([pos.x0 + shift, pos.y0, pos.width, pos.height])
    axes[5].grid(alpha=0.2)
    axes[5].set_ylim(top = 0.5)

    titles = ["sqrt(FWHM^2-(Blur^2-Cam^2))", "FWHM:sqrt(95^2-5^2)", "sqrt(AOS^2+Cam^2)"]
    passes = [pass_2, pass_3, pass_5]
    for ax, t, p in zip([axes[2], axes[3], axes[5]], titles, passes):
        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()
    
        x_hi = min(0.45, xmax)
        y_hi = min(0.1, ymax)
    
        ax.fill_between(
            [xmin, x_hi],
            ymin, y_hi,
            color=(0.45, 0.85, 0.45),
            alpha=0.15,
            zorder=0
        )
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)

        ax.text(
            (xmin + xmax) / 2,
            0.9*ymax,
            t + p,
            ha='center', va='center',
            color='black',
            alpha=0.8,
            zorder=2
        )

    my_suptitle = fig.suptitle("Survey Mode Performance and AOS DoFs – Day " + str(day_obs), fontsize=18, x=0.4, y=1.03)
else:
    print('No data to plot')
fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/Full_Night_Report_{day_obs}.png", 
            bbox_inches='tight', pad_inches=1.2, bbox_extra_artists=[my_suptitle])

In [ ]:
filtered_table.columns

In [ ]:
fbs_table = filtered_table[(filtered_table['seq'] > 327)]
fbs_table['psf_ellipticity_median'].median()

In [ ]:
len(fbs_table)

In [ ]:
fbs_table['aos_fwhm'].median()

In [ ]:
fbs_table['psf_fwhm'].median()

In [ ]:
db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
await db.create(simplified=False)


In [ ]:
from lsst.summit.utils.efdUtils import calcNextDay
startDay = 20251115
endDay = 20260115
day_obs = startDay
day_obss = []
aos_fwhms = []
corr_lags = []
while day_obs <= endDay:
    try:
        db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
        await db.create(simplified=False)
        table = db.table
        if len(table)>0:
            filtered_table = table[table['day_obs'] == day_obs]
            raw_filtered_table = table[table['day_obs'] == day_obs]
            filtered_table = filtered_table.select_dtypes(include="number")
            filtered_table['band'] = raw_filtered_table['band']
            filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col == 'band' else 'mean' for col in filtered_table.columns})
        else:
            day_obs = calcNextDay(day_obs)
            continue
    except:
        day_obs = calcNextDay(day_obs)
        continue
        
    max_corr_lag = 10
    corr_lag = filtered_table['seq'] - filtered_table['seq_num_corr']
    corr_lag = np.clip(corr_lag, 0, max_corr_lag)
    corr_lag = np.nanmean(corr_lag)
    aos_fwhm = np.nanmedian(filtered_table["aos_fwhm"].values)
    print(day_obs, aos_fwhm, corr_lag)
    aos_fwhms.append(aos_fwhm)
    corr_lags.append(corr_lag)
    day_obss.append(day_obs)
    day_obs = calcNextDay(day_obs)


In [ ]:
plt.scatter(corr_lags, aos_fwhms)
plt.ylim(0.3, 0.8)
plt.xlim(0, 10)
fit = np.polyfit(corr_lags, aos_fwhms, 1)
xs = np.linspace(0,10,100)
ys = xs * fit[0] + fit[1]
plt.plot(xs, ys, ls='--', color='red')
plt.xlabel("Mean correction lag (images)")
plt.ylabel("Median AOS_FWHM")
#plt.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/AOS_FWHM_vs_Corr_Lag_16Jan26.png")

In [ ]:
aos_fwhms2 = []
corr_lags2 = []
z4s = []
z11s = []
z4stds = []
z11stds = []
for filtered_table in filtered_tables:        
    max_corr_lag = 10
    corr_lag = filtered_table['seq'] - filtered_table['seq_num_corr']
    corr_lag = np.clip(corr_lag, 0, max_corr_lag)
    corr_lag = np.nanmean(corr_lag)
    aos_fwhm = np.nanmedian(filtered_table["aos_fwhm"].values)
    z4 = np.nanmedian(filtered_table["z4"].values)
    z11 = np.nanmedian(filtered_table["z11"].values)
    z4std = np.nanstd(filtered_table["z4"].values)
    z11std = np.nanstd(filtered_table["z11"].values)
    #print(aos_fwhm, corr_lag)
    aos_fwhms2.append(aos_fwhm)
    corr_lags2.append(corr_lag)
    z4s.append(z4)
    z11s.append(z11)
    z4stds.append(z4std)
    z11stds.append(z11std)
print(len(dayObss), len(aos_fhms2), len(corr_lags2))

In [ ]:
%matplotlib inline
fig = plt.figure(figsize=(12,6))
xaxis = list(range(len(day_obss)))
#xaxis = list(range(41))
stucks = [14, 27, 31]
gs = gridspec.GridSpec(
    nrows=2, ncols=1,
    width_ratios=[4],
    height_ratios=[1]*2,
    hspace=0.0,
    top=0.9,
    bottom=0.3
)
axes = []
for i in range(2):
    ax = fig.add_subplot(gs[i], sharex=axes[0] if i > 0 else None)
    ax.set_xticks(xaxis)
    ax.set_xticklabels(day_obss, rotation=90)
    axes.append(ax)
axes[0].scatter(xaxis, aos_fwhms)
#axes[0].scatter(stucks, [aos_fwhms[i] for i in stucks], marker='o', facecolors='none', edgecolors='red', s=100, label="Stuck at 0.5")
axes[0].set_ylim(0.2, 0.8)
axes[0].set_ylabel("AOS_FWHM")
axes[0].legend()
axes[0].axhline(0.5, ls='--', color='black')
axes[1].scatter(xaxis, corr_lags)
#axes[1].scatter(stucks, [corr_lags[i] for i in stucks], marker='o', facecolors='none', edgecolors='red', s=100)
axes[1].set_ylim(0,10)
axes[1].set_ylabel("Corr Lag(images)")
plt.suptitle("AOS_FWHM and Corr Lag trends", y=0.95)
fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/AOS_FWHM_Corr_Lag_Trends_16Jan26.png")

In [ ]:
%matplotlib inline
fig = plt.figure(figsize=(12,6))
xaxis = list(range(len(day_obss)))
gs = gridspec.GridSpec(
    nrows=2, ncols=1,
    width_ratios=[4],
    height_ratios=[1]*2,
    hspace=0.0,
    top=0.9,
    bottom=0.3
)
axes = []
for i in range(2):
    ax = fig.add_subplot(gs[i], sharex=axes[0] if i > 0 else None)
    ax.set_xticks(xaxis)
    ax.set_xticklabels(day_obss, rotation=90)
    axes.append(ax)
axes[0].errorbar(xaxis, z4s, yerr=z4stds)
axes[0].axhline(0.0, ls='--', color='black')
axes[0].axhline(-0.2, ls='--', color='green')
axes[0].set_ylim(-1.0, 1.0)
axes[0].set_ylabel("Z4")
axes[0].legend()
axes[1].errorbar(xaxis, z11s, yerr=z11stds)
axes[1].axhline(0.0, ls='--', color='black')
axes[1].set_ylim(-0.5, 0.50)
axes[1].set_ylabel("Z11")
plt.suptitle("Z4 and Z11 trends", y=0.95)
fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/Z4_Z11_Trends_30Dec25.png")

In [ ]:
table.columns

In [ ]:
from lsst.summit.utils.efdUtils import calcNextDay
startDay = 20251115
endDay = 20260104
day_obs = startDay
day_obss = []
filtered_tables = []
while day_obs <= endDay:
    try:
        db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
        await db.create(simplified=False)
        table = db.table
        if len(table)>0:
            filtered_table = table[table['day_obs'] == day_obs]
            raw_filtered_table = table[table['day_obs'] == day_obs]
            filtered_table = filtered_table.select_dtypes(include="number")
            filtered_table['band'] = raw_filtered_table['band']
            filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col == 'band' else 'mean' for col in filtered_table.columns})
        else:
            day_obs = calcNextDay(day_obs)
            continue
    except:
        day_obs = calcNextDay(day_obs)
        continue
        
    filtered_tables.append(filtered_table)
    day_obss.append(day_obs)
    print(f"Finished {day_obs}")
    day_obs = calcNextDay(day_obs)

import pickle as pkl
filename = f"/home/c/cslage/u/MTAOS/times_square_reports/filtered_tables_{startDay}_{endDay}.pkl"
with open(filename, 'wb') as f:
    pkl.dump([day_obss, filtered_tables], f)


In [ ]:
import pickle as pkl
startDay = 20251115
endDay = 20260104

filename = f"/home/c/cslage/u/MTAOS/times_square_reports/filtered_tables_{startDay}_{endDay}.pkl"
with open(filename, 'rb') as f:
    [day_obss, filtered_tables] = pkl.load(f)

len(day_obss)

In [ ]:
filtered_tables[0].columns

In [ ]:
filtered_tables[0]['rotation_angle']

In [ ]:
stucks = [14, 27, 31]

from matplotlib.backends.backend_pdf import PdfPages
pdf = PdfPages(f"/home/c/cslage/u/MTAOS/times_square_reports/Correlation_{startDay}_{endDay}.pdf")
fig = plt.figure(figsize=(10, 10))
for col in filtered_tables[0].columns:
    try:
        aos_fwhms = []
        xaxis = []
        stuck_y = []
        stuck_x = []
        for i, filtered_table in enumerate(filtered_tables):
            try:
                aos_fwhm = np.nanmedian(filtered_table['aos_fwhm'].values)
                xax = np.nanmedian(filtered_table[col].values)
            except: 
                continue
            aos_fwhms.append(aos_fwhm)
            xaxis.append(xax)
            if i in stucks:
                stuck_y.append(aos_fwhm)
                stuck_x.append(xax)
                

        plt.scatter(xaxis, aos_fwhms)
        plt.scatter(stuck_x, stuck_y, marker='o', facecolors='none', edgecolors='red', s=100, label="Stuck at 0.5")
        plt.ylabel("AOS_FWHM")
        plt.xlabel(col)
        plt.title(col, fontsize=18)
        plt.legend()
        pdf.savefig(fig)
        fig.clf()
        print(f"Finished {col}")
    except:
        print(f"{col} failed to plot")
        continue
pdf.close()    
    

In [ ]:
import mpl_scatter_density
from matplotlib.backends.backend_pdf import PdfPages
pdf = PdfPages(f"/home/c/cslage/u/MTAOS/times_square_reports/Correlation_Density_{startDay}_{endDay}.pdf")
fig = plt.figure(figsize=(10,10))

from matplotlib.colors import LinearSegmentedColormap

# "Viridis-like" colormap with white background
white_viridis = LinearSegmentedColormap.from_list('white_viridis', [
    (0, '#ffffff'),
    (1e-20, '#440053'),
    (0.2, '#404388'),
    (0.4, '#2a788e'),
    (0.6, '#21a784'),
    (0.8, '#78d151'),
    (1, '#fde624'),
], N=256)


for col in filtered_tables[0].columns:
    try:
        aos_fwhms = []
        xaxis = []
        ax = fig.add_subplot(1, 1, 1, projection='scatter_density')
        for i, filtered_table in enumerate(filtered_tables):
            for j in range(len(filtered_table)):
                try:
                    aos_fwhm = filtered_table['aos_fwhm'].values[j]
                    xax = filtered_table[col].values[j]
                except: 
                    continue
                if np.isnan(aos_fwhm) or np.isnan(xax):
                    continue
                aos_fwhms.append(aos_fwhm)
                xaxis.append(xax)  
        sp = ax.scatter_density(xaxis, aos_fwhms, cmap=white_viridis)
        fig.colorbar(sp, label='Number of points per pixel')
        ax.set_ylabel("AOS_FWHM")
        ax.set_xlabel(col)
        ax.set_title(col, fontsize=18)
        ax.legend()
        pdf.savefig(fig)
        fig.clf()
        print(f"Finished {col}")
    except:
        print(f"{col} failed to plot")
        continue
pdf.close()    
    